# 06 — Model Comparison and Leaderboard

This notebook:
1. Loads all predictions from NB04–06
2. Builds a complete leaderboard (A1, A2, B1, B2, C1, C2)
3. Computes pairwise comparisons vs B2 baseline
4. Saves `results/leaderboard.csv`

**Design rule:** test split is used once here, final evaluation only.

In [1]:
import matplotlib; matplotlib.use('Agg')
import json as _json, os, warnings
from pathlib import Path

if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir(os.path.dirname(os.getcwd()))

import sys
if "src" not in sys.path[0]:
    sys.path.insert(0, "src")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
import joblib
warnings.filterwarnings("ignore")

from creator_analytics import features as fe
from creator_analytics.config import load_config
from creator_analytics.evaluate import compute_metrics, paired_comparison

cfg       = load_config("config/default.yaml")
PROCESSED = Path(cfg["paths"]["processed_dir"])
PREDS_DIR = Path("results/predictions")
FIGS      = Path("results/figures"); FIGS.mkdir(parents=True, exist_ok=True)
SEED      = cfg["seed"]


In [2]:
data, spec = fe.load_feature_table(PROCESSED)
idx       = pd.read_parquet(PROCESSED / "dataset_index.parquet")
TEST      = idx["split"].eq("test")
auth_test = idx.loc[TEST, "author_id"].reset_index(drop=True)

nb04 = pd.read_parquet(PREDS_DIR / "nb04_predictions.parquet")
nb06 = pd.read_parquet(PREDS_DIR / "nb05_predictions.parquet")

y_test     = nb04["y_true"]
zero_preds = np.zeros(len(y_test))

META       = ["video_id", "author_id", "y_true"]
cols04     = [c for c in nb04.columns if c not in META]
cols06     = [c for c in nb06.columns if c not in META and c != "B2_global_XGB"]

all_preds: dict[str, np.ndarray] = {}
for c in cols04:
    all_preds[c] = nb04[c].values
for c in cols06:
    all_preds[c] = nb06[c].values

print("Models loaded:", list(all_preds.keys()))
print(f"Test videos: {len(y_test):,}  |  Creators: {auth_test.nunique():,}")


Models loaded: ['A1_zero', 'A2_creator_mean', 'B1_RF', 'B2_XGB', 'C1_global_LGBM', 'C2_global_Ridge']
Test videos: 40,124  |  Creators: 1,437


## Leaderboard

In [3]:
# Load training times
_times_path = Path("models/alternative/training_times.json")
_train_times: dict = _json.loads(_times_path.read_text()) if _times_path.exists() else {}

# Parse A/B times from experiments.md
import re as _re
_exp_path = Path("results/experiments.md")
if _exp_path.exists():
    for line in _exp_path.read_text().splitlines():
        if not line.startswith("| 20"):
            continue
        parts = [p.strip() for p in line.split("|")[1:-1]]
        if len(parts) < 11:
            continue
        exp_name, time_col = parts[1], parts[9]
        _key_map = {
            "A1_predict_zero": "A1_zero", "A2_creator_mean": "A2_creator_mean",
            "B1_global_RF": "B1_RF", "B2_global_XGB": "B2_XGB",
        }
        pred_key = _key_map.get(exp_name)
        if pred_key and pred_key not in _train_times:
            m = _re.match(r"~?(\d+(?:\.\d+)?)s", time_col)
            if m:
                _train_times[pred_key] = float(m.group(1))

b2_preds = all_preds["B2_XGB"]
rows = []
for name, preds in all_preds.items():
    nan_n = int(np.isnan(preds).sum())
    pc = preds.copy()
    if nan_n > 0:
        pc[np.isnan(pc)] = b2_preds[np.isnan(pc)]
    m        = compute_metrics(y_test, pc)
    cmp_b2   = paired_comparison(y_test, pc, b2_preds, auth_test, name, "B2_XGB")
    cmp_zero = paired_comparison(y_test, pc, zero_preds, auth_test, name, "predict-0")
    rows.append({
        "model":        name,
        "rmse":         m["rmse"],
        "mae":          m["mae"],
        "r2":           m["r2"],
        "spearman":     m["spearman"],
        "train_time_s": _train_times.get(name),
        "vs_B2_diff":   cmp_b2["rmse_diff"],
        "vs_B2_p":      cmp_b2["wilcoxon_pvalue"],
        "vs_B2_win":    round(cmp_b2["win_rate_a"], 3),
        "vs_zero_diff": cmp_zero["rmse_diff"],
        "vs_zero_p":    cmp_zero["wilcoxon_pvalue"],
        "nan_preds":    nan_n,
    })

leaderboard = pd.DataFrame(rows).sort_values("rmse").reset_index(drop=True)
leaderboard.index += 1
display(leaderboard.round(4))
leaderboard.to_csv("results/leaderboard.csv", index_label="rank")
print("Saved results/leaderboard.csv")


,model,rmse,mae,r2,spearman,train_time_s,vs_B2_diff,vs_B2_p,vs_B2_win,vs_zero_diff,vs_zero_p,nan_preds
1,C1_global_LGBM,0.9104,0.5841,0.1541,0.3198,NaN,-0.0022,0.0027,0.507,-0.0900,0.0,0
2,B1_RF,0.9113,0.5789,0.1525,0.3264,845.0,-0.0014,0.0000,0.525,-0.0892,0.0,0
3,B2_XGB,0.9127,0.5855,0.1499,0.3101,900.0,0.0000,NaN,0.000,-0.0878,0.0,0
4,A2_creator_mean,0.9690,0.6339,0.0418,0.0759,0.0,0.0563,0.0000,0.440,-0.0315,0.0,0
5,C2_global_Ridge,0.9718,0.6182,0.0362,0.1416,NaN,0.0591,0.0000,0.471,-0.0287,0.0,0
6,A1_zero,1.0005,0.6030,-0.0215,NaN,0.0,0.0878,0.0000,0.512,0.0000,NaN,0


Saved results/leaderboard.csv


**Result:** Leaderboard: C1 LGBM 0.9104 → B1 RF 0.9113 → B2 XGB 0.9127 → A2 creator-mean 0.9690 → C2 Ridge 0.9718 → A1 predict-0 1.0005.
**Takeaway:** The three tree models are within 0.002 RMSE of each other — differences in the third decimal place. They beat both trivial baselines by ~0.09 RMSE. No single tree model decisively dominates; any of the three can be used.

## Pairwise comparisons vs B2 baseline

In [4]:
print("=== Model comparisons vs B2 global XGB ===")
for name in ["A1_zero", "A2_creator_mean", "B1_RF", "C1_global_LGBM", "C2_global_Ridge"]:
    if name not in all_preds:
        continue
    cmp = paired_comparison(y_test, all_preds[name], b2_preds, auth_test, name, "B2_XGB")
    print(f"  {name:22s}  RMSE_diff={cmp['rmse_diff']:+.4f}  "
          f"win_rate={cmp['win_rate_a']:.3f}  p={cmp['wilcoxon_pvalue']:.4f}")


=== Model comparisons vs B2 global XGB ===


  A1_zero                 RMSE_diff=+0.0878  win_rate=0.512  p=0.0000


  A2_creator_mean         RMSE_diff=+0.0563  win_rate=0.440  p=0.0000


  B1_RF                   RMSE_diff=-0.0014  win_rate=0.525  p=0.0000


  C1_global_LGBM          RMSE_diff=-0.0022  win_rate=0.507  p=0.0027


  C2_global_Ridge         RMSE_diff=+0.0591  win_rate=0.471  p=0.0000


**Result:** B1 RF and C1 LGBM both beat B2 XGB (p < 0.003). Win rates vs B2: B1 RF 52.5 %, C1 LGBM 50.7 %. RMSE differences: −0.0014 and −0.0022 respectively.
**Takeaway:** Statistical significance does not mean practical significance: all three models produce the same prediction on roughly half of all videos. RF and LGBM are preferred over XGB at no meaningful accuracy cost.

## Visual comparison

In [5]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

top = leaderboard.head(min(8, len(leaderboard)))

ax = axes[0]
ax.barh(top["model"][::-1], top["rmse"][::-1], color="steelblue")
ax.axvline(leaderboard.loc[leaderboard["model"] == "B2_XGB", "rmse"].values[0],
           color="red", linestyle="--", label="B2 XGB")
ax.set_xlabel("Test RMSE (lower is better)")
ax.set_title("Models by RMSE")
ax.legend(fontsize=8)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

ax = axes[1]
ax.barh(top["model"][::-1], top["vs_B2_win"][::-1],
        color=["#2E7D32" if w > 0.5 else "#C62828" for w in top["vs_B2_win"][::-1]])
ax.axvline(0.5, color="black", linestyle="--")
ax.set_xlabel("Win rate vs B2 XGB (per-video)")
ax.set_title("Win Rate vs B2 Baseline")
ax.set_xlim(0, 1)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.tight_layout()
fig.savefig(FIGS / "nb06_leaderboard.png", dpi=120, bbox_inches="tight")
plt.show()
print("Saved nb06_leaderboard.png")


Saved nb06_leaderboard.png


## SHAP importance (global model)

In [6]:
shap_path = Path("results/shap_global_test.parquet")
if not shap_path.exists():
    print("shap_global_test.parquet not found — skipping SHAP section.")
    print("Re-execute NB07 after generating SHAP values.")
else:
    shap_df = pd.read_parquet(shap_path)
    feat_imp = (
        shap_df.abs().mean()
        .sort_values(ascending=False)
        .head(20)
        .reset_index()
    )
    feat_imp.columns = ["feature", "mean_abs_shap"]

    fig, ax = plt.subplots(figsize=(7, 7))
    ax.barh(feat_imp["feature"][::-1], feat_imp["mean_abs_shap"][::-1], color="#1565C0")
    ax.set_xlabel("Mean |SHAP|")
    ax.set_title("Global XGB — Top-20 Features by SHAP Importance")
    ax.tick_params(axis="y", labelsize=8)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    plt.tight_layout()
    fig.savefig(FIGS / "nb06_shap_global.png", dpi=120, bbox_inches="tight")
    plt.show()
    print("Saved nb06_shap_global.png")
    print(feat_imp.head(10).to_string(index=False))


Saved nb06_shap_global.png
               feature  mean_abs_shap
    follower_count_log       0.074607
      sticker_text_len       0.066211
          duration_log       0.049419
                   joy       0.045743
               disgust       0.042397
              desc_len       0.039111
video_count_at_publish       0.037220
         speaking_rate       0.036318
 days_since_prev_video       0.028936
   music_selected_from       0.027974


**Result:** Top SHAP features (mean |SHAP| on 40k test videos): `follower_count_log` (0.075), `sticker_text_len` (0.066), `duration_log` (0.049), `joy` (0.046), `disgust` (0.042).
**Takeaway:** Account size (`follower_count_log`) is the single most influential feature — consistent with the ablation finding that `creator_context` carries most signal. The strongest content-side features are on-screen text length and video duration; emotion scores appear but rank below account-level features.

## Summary

*(See `results/leaderboard.csv` for full table)*

**Rankings by test RMSE (lower is better):**

| Rank | Model | Test RMSE | vs B2 (XGB) |
|------|-------|-----------|-------------|
| 1 | C1 global LGBM | 0.9104 | −0.0022 |
| 2 | B1 global RF   | 0.9113 | −0.0014 |
| 3 | B2 global XGB  | 0.9127 | baseline |
| 4 | A2 creator-mean | 0.9690 | +0.0563 |
| 5 | C2 global Ridge | 0.9718 | +0.0591 |
| 6 | A1 predict-0 | 1.0005 | +0.0878 |

**Conclusions:**
- Differences between C1, B1, B2 are within the third decimal place (< 0.003 RMSE) — no single model decisively dominates.
- All three tree models beat both trivial baselines (A1, A2) by a wide margin.
- Ridge (C2) is slower than tree models to converge and slightly worse, but interpretable.
- Top SHAP features: `follower_count_log`, `sticker_text_len`, `duration_log`.
